# Temporal-weight sensitivity of the delta tilt method

Compare **equal**, **Gaussian σ = 1.5 days**, and **Gaussian σ = 1 day** temporal weighting on **random real eddy-day snapshots**, then follow each sampled eddy through its available lifetime. The Gaussian 1.5-day kernel gives the reference day **27.1%** of the weight when all seven daily increments are available.

Only the temporal mean of each depth increment changes. The existing **unweighted sample variance across days**, inverse-variance depth weighting, depth interpolation, 3-D line fit and bearing convention remain unchanged. This is a sensitivity experiment; it does not update the production dataset.

Run all cells on Katana. Change `SEED` to choose another sample. Optional `EDDY_IDS` restricts the sample to eddies of interest. Small helper functions are in `delta_sensitivity_tools.py` to keep this notebook short.

**Depth correction:** all three methods reconstruct and fit only intervals supported by the reference day. This applies to the snapshot and lifetime plots; neighbouring days cannot extend the reference profile. The production pipeline remains unchanged.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = next(p for p in (HERE, *HERE.parents) if (p / 'seacofs_eddy_dataset_modular').is_dir())
FOLDER = REPO / 'seacofs_eddy_tilt_analysis' / 'delta_tilt_method'
sys.path.insert(0, str(FOLDER))
from delta_sensitivity_tools import METHODS, increments, fit_snapshot, maximum_span, lifetime

PROFILE_PATH = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/vertical_profiles_confirmed/profiles.parquet')
SEED = 729
N_SNAPSHOTS = 20
EDDY_IDS = None             # e.g. [123, 456]; None searches all eddies
MAX_DEPTH = 1000
DEPTH_INT = 10
SAVE = False                # Optional PNG figures and CSV tables
OUT = FOLDER / 'sensitivity_outputs'
# COLORS = {'Equal': '#222222', 'Gaussian 2 d': '#1f77b4', 'Gaussian 1.5 d': '#d95f02'}
COLORS = {
    'Equal': '#222222',
    'Gaussian 1.5 d': '#1f77b4',
    'Gaussian 1.0 d': '#d95f02',
}
if SAVE:
    OUT.mkdir(exist_ok=True)

In [ ]:
offsets = np.arange(-3, 4)
kernels = {}
for name, sigma in METHODS.items():
    a = np.ones(7) if sigma is None else np.exp(-0.5*(offsets/sigma)**2)
    kernels[name] = 100*a/a.sum()
display(pd.DataFrame(kernels, index=offsets).rename_axis('Days from reference').round(1))

## Choose snapshots

Random sampling is without replacement among eligible **eddy-days**, so longer eddies can contribute more snapshots. To keep the schematic clear, all seven daily profiles must have at least 200 m of usable increment-depth span and enough depth points. Each selected snapshot must also yield all three fits. This is an illustrative sample, not a population test. The lifetime plots subsequently retain gaps and incomplete windows.

In [ ]:
profiles = pd.read_parquet(PROFILE_PATH, columns=['Eddy', 'Day', 'Depth', 'xc', 'yc'])
if EDDY_IDS is not None:
    profiles = profiles.loc[profiles.Eddy.isin(EDDY_IDS)].copy()
profiles['Depth'] = profiles.Depth.abs()
if profiles[['Eddy', 'Day']].isna().any().any():
    raise ValueError('Missing Eddy or Day identifiers.')
if profiles.duplicated(['Eddy', 'Day', 'Depth']).any():
    raise ValueError('Duplicate Eddy-Day-Depth rows must be resolved first.')
if not np.isfinite(profiles[['Depth', 'xc', 'yc']].to_numpy()).all():
    raise ValueError('Nonfinite profile coordinates must be resolved first.')
limited = profiles.loc[profiles.Depth.le(MAX_DEPTH)]
coverage = limited.groupby(['Eddy', 'Day']).Depth.agg(['min', 'max', 'size'])
coverage['first'] = np.ceil(coverage['min']/DEPTH_INT)*DEPTH_INT
coverage['last'] = np.floor(coverage['max']/DEPTH_INT)*DEPTH_INT - DEPTH_INT
eligible = coverage.loc[(coverage['size'] >= 2) & ((coverage['last']-coverage['first']) >= 200)
                        & ((coverage['last']-coverage['first'])/DEPTH_INT + 1 >= 5)]
candidates = []
for eddy, g in eligible.groupby(level='Eddy'):
    valid = set(g.index.get_level_values('Day').astype(int))
    centres = set(valid)
    for offset in range(-3, 4):
        centres &= {day-offset for day in valid}
    candidates.extend((int(eddy), day) for day in sorted(centres))
rng = np.random.default_rng(SEED)
selected, sample_fits = [], {}
for idx in rng.permutation(len(candidates)):
    eddy, day = candidates[idx]
    g = profiles.loc[profiles.Eddy.eq(eddy) & profiles.Day.between(day-3, day+3)]
    dx, dy = increments(g, DEPTH_INT, MAX_DEPTH)
    fits = {name: fit_snapshot(dx, dy, day, sigma) for name, sigma in METHODS.items()}
    if any(f is None for f in fits.values()) or fits['Equal']['TiltDis'] < 1e-10:
        continue
    selected.append((eddy, day))
    sample_fits[(eddy, day)] = fits
    if len(selected) == N_SNAPSHOTS:
        break
if len(selected) != N_SNAPSHOTS:
    raise ValueError(f'Only {len(selected)} valid snapshots; reduce N_SNAPSHOTS or broaden EDDY_IDS.')
samples = pd.DataFrame(selected, columns=['Eddy', 'Day'])
display(samples)
print(f'{len(samples)} snapshots from {samples.Eddy.nunique()} eddies; seed {SEED}')

## Overlay the snapshot reconstructions and fits

Each figure uses the **equal-weight fit's horizontal axis for all curves**. Panel a shows the seven actual centre profiles; b overlays reconstructed profiles (solid) and fitted lines (dashed); c overlays fitted lines and the reference-day profile. All reconstructed curves share the same display translation, so genuine differences between methods remain visible.

The legends report **full 2-D fitted tilt distance**, which can exceed its horizontal span in this common projection if a fit turns away from the viewing axis. `Maximum span` is the maximum **pairwise Euclidean horizontal distance** between any two fitted centres on the reference day within 0–1000 m; it is neither a projected range nor necessarily the surface-to-deep distance. It can occur between intermediate depths. The raw profile in c is separately referenced to its shallowest centre, as in the original schematic.

In [ ]:
summary = []
for eddy, day in selected:
    fits = sample_fits[(eddy, day)]
    base = fits['Equal']
    axis = base['ends'][1, :2] - base['ends'][0, :2]
    axis = axis / np.linalg.norm(axis)
    origin = base['trace'][['x', 'y']].iloc[0].to_numpy() @ axis
    window = limited.loc[limited.Eddy.eq(eddy) & limited.Day.between(day-3, day+3)]
    ref = window.loc[window.Day.eq(day)].sort_values('Depth')
    span = maximum_span(ref, MAX_DEPTH)
    reference_bottom = ref.Depth.max()
    fig, axs = plt.subplots(1, 3, figsize=(14, 4.4), sharey=True, constrained_layout=True)
    fig.suptitle(f'Eddy {eddy} · Day {day} · maximum 2-D span = {span:.2f} km')
    for d, g in window.groupby('Day'):
        g = g.sort_values('Depth')
        axs[0].plot(g[['xc', 'yc']].to_numpy() @ axis, g.Depth,
                    color='crimson' if d == day else plt.cm.viridis((d-day+3)/6),
                    lw=2.5 if d == day else 1, label=f'k{int(d-day):+d}' if d != day else 'k')
    for name, f in fits.items():
        tr, ends = f['trace'], f['ends']
        assert tr.Depth.min() >= ref.Depth.min()
        assert tr.Depth.max() + DEPTH_INT <= reference_bottom + 1e-8
        s = tr[['x', 'y']].to_numpy() @ axis - origin
        e = ends[:, :2] @ axis - origin
        label = f"{name}: {f['TiltDis']:.2f} km"
        axs[1].plot(s, tr.Depth, color=COLORS[name], label=label)
        for ax in axs[1:]:
            ax.plot(e, ends[:, 2], '--', color=COLORS[name], lw=2,
                    label=label if ax is axs[2] else None)
        summary.append({'Eddy': eddy, 'Day': day, 'Method': name,
                        'TiltDis': f['TiltDis'], 'TiltDir': f['TiltDir'], 'Maximum span': span,
                        'ReferenceDepthMax': reference_bottom, 'FitDepthMax': ends[1, 2],
                        'Change from equal (km)': f['TiltDis']-base['TiltDis'],
                        'Direction change (deg)': (f['TiltDir']-base['TiltDir']+180)%360-180})
    xy = ref[['xc', 'yc']].to_numpy()
    axs[2].plot((xy-xy[0]) @ axis, ref.Depth, color='crimson', lw=1.7, label='Day k profile')
    for ax, title in zip(axs, ['a) Daily profiles', 'b) Reconstruction + fit', 'c) Fits + reference day']):
        ax.set_title(title, loc='left', fontsize=11)
        ax.set_ylim(MAX_DEPTH, 0)
        ax.axhline(reference_bottom, color='grey', ls=':', lw=1, alpha=0.7)
        ax.grid(alpha=0.15)
        ax.legend(fontsize=8, loc='lower left')
    axs[0].set_ylabel('Depth (m)')
    axs[0].set_xlabel('Projected centre position (km)')
    for ax in axs[1:]:
        ax.set_xlabel('Displacement on common axis (km)')
    common = [ax.get_xlim() for ax in axs[1:]]
    for ax in axs[1:]:
        ax.set_xlim(min(x[0] for x in common), max(x[1] for x in common))
    if SAVE:
        fig.savefig(OUT / f'snapshot_{eddy}_{day}.png', dpi=180)
    plt.show()
    plt.close(fig)
summary = pd.DataFrame(summary)
display(summary.round(2))

## Compare methods over each sampled eddy's lifetime

One figure per unique sampled eddy. The upper panel compares all four distances; the lower panel isolates each Gaussian estimate's change from equal weighting. Vertical dotted lines mark the sampled days. The x-axis is days since the first available confirmed profile, rather than necessarily the eddy's physical formation.

All methods use profiles within 0–1000 m. Maximum span uses the actual centres on that day; delta estimates use smoothed increments restricted to the reference day’s depth support, with the existing upper-interval labels. Thus differences from maximum span include smoothing, curvature, depth coverage and fitting—not only temporal weights. The first/last three days have no delta estimate, matching the legacy seven-day settings. Missing reference days now remain gaps in **all** curves: neighbouring profiles cannot supply an estimate for an absent reference day. Where increments are missing, temporal weights renormalize over available days at each depth. A 27.1% central weight applies only where all seven increments are present; intervals without a valid central increment are excluded entirely.

The unweighted sample variance and minimum fit-depth criteria are unchanged. The reference limit is applied before the cumulative sum, so neighbouring increments outside the range cannot affect the reconstructed profile. The legacy production-equality check has been removed because this intentionally corrected estimate can differ from production.


In [ ]:
lifetimes = []
for eddy in samples.Eddy.unique():
    track = profiles.loc[profiles.Eddy.eq(eddy)].copy()
    table, _, _ = lifetime(track, DEPTH_INT, MAX_DEPTH)
    table['Eddy'] = eddy
    lifetimes.append(table)
    age = table.Day - table.Day.min()
    fig, axs = plt.subplots(2, 1, figsize=(11, 5), sharex=True, constrained_layout=True,
                            gridspec_kw={'height_ratios': [2, 1]})
    axs[0].plot(age, table['Maximum span'], color='crimson', alpha=0.8, label='Maximum 2-D span (daily)')
    for name in METHODS:
        # axs[0].plot(age, table[name], color=COLORS[name], label=name,
        #             ls={'Equal': '-', 'Gaussian 2 d': '--', 'Gaussian 1.5 d': ':'}[name], lw=1.8)
        axs[0].plot(age, table[name], color=COLORS[name], label=name,
                    ls={'Equal': '-', 'Gaussian 1.5 d': '--', 'Gaussian 1.0 d': ':'}[name], lw=1.8)
        if name != 'Equal':
            axs[1].plot(age, table[name]-table['Equal'], color=COLORS[name], label=name)
    for d in samples.loc[samples.Eddy.eq(eddy), 'Day']:
        for ax in axs:
            ax.axvline(d-table.Day.min(), color='grey', ls=':', alpha=0.5)
    axs[0].set_title(f'Eddy {eddy}: temporal-weight sensitivity')
    axs[0].set_ylabel('Tilt distance / span (km)')
    axs[1].set_ylabel('Change from equal (km)')
    axs[1].set_xlabel('Days since first available profile')
    axs[1].axhline(0, color='grey', lw=0.8)
    for ax in axs:
        ax.legend(fontsize=9, ncol=2)
        ax.grid(alpha=0.15)
    if SAVE:
        fig.savefig(OUT / f'lifetime_{eddy}.png', dpi=180)
    plt.show()
    plt.close(fig)
lifetimes = pd.concat(lifetimes, ignore_index=True)
print('All delta lifetime estimates use reference-day depth limits; production was not changed.')
if SAVE:
    samples.to_csv(OUT / 'selected_snapshots.csv', index=False)
    summary.to_csv(OUT / 'snapshot_comparison.csv', index=False)
    lifetimes.to_csv(OUT / 'lifetime_comparison.csv', index=False)

**What to look for:** Does Gaussian 1.5-day weighting respond more clearly to coherent changes visible in the daily profiles, or mainly add day-to-day variability? Is the difference primarily in distance, direction, or both? The maximum-span curve is a geometric comparison, not ground truth for delta tilt. Agreement with it alone does not establish the best kernel. These sampled examples support visual sensitivity assessment, not statistical significance or population-wide conclusions.